💡 **Environment:** `clamp-analyses`

# Description

**Sandbox NB 00/03 — observed (un-adjusted) drug-disease scores for lipid diseases.**

Computes the *current* pipeline score `score = -1 * drugᵀ · disease` for the **single-gene** and
**module-based (ARCHS4)** methods, restricted to the simplest case: the **Liver** tissue and the
**all-genes / all-LVs** threshold (no top-N masking).

It then maps trait columns → DOID (reusing `libs.drug_disease_utils.map_traits_to_doid`), keeps the
two lipid DOIDs that exist in the PharmacotherapyDB gold standard, lists the current top-scoring
drugs, and records where the gold-standard positives/negatives land. Outputs feed NB01 (null
adjustment) and NB02 (comparison).

See this directory's `CLAUDE.md` for scope and the null-adjustment rationale.

# Modules loading

In [1]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [2]:
DATA_DIR = here('data/drug_disease_associations')
assert DATA_DIR.exists()

# Read-only inputs (single-gene: gene-level matrices; module: CLAMP-projected matrices)
LINCS_RAW_FILE = DATA_DIR / 'lincs-data.pkl'
SPREDIXCAN_RAW_LIVER = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '00_spredixcan_projection_archs4/spredixcan/raw/'
    'spredixcan-mashr-zscores-Liver-data.pkl')
LINCS_PROJ_FILE = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '01_lincs_projection_archs4/lincs/lincs-projection.pkl')
SPREDIXCAN_PROJ_LIVER = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    '00_spredixcan_projection_archs4/spredixcan/proj/'
    'spredixcan-mashr-zscores-Liver-projection-archs4.pkl')
for f in (LINCS_RAW_FILE, SPREDIXCAN_RAW_LIVER, LINCS_PROJ_FILE, SPREDIXCAN_PROJ_LIVER):
    assert f.exists(), f

# The two lipid DOIDs present in the gold standard (see CLAUDE.md)
LIPID_DOIDS = ['DOID:1936', 'DOID:3393']

METHODS = ['gene_based', 'module_based_archs4']

OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/null_adjust_test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/null_adjust_test')

# Load gold standard + human-readable names

In [3]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
doids_in_gold_standard = set(gold_standard['trait'])
print(f'gold standard: {gold_standard.shape}, {len(doids_in_gold_standard)} DOIDs')

# DOID -> disease name and DrugBank id -> drug name (for readable tables)
_ind = pd.read_csv(DATA_DIR / 'indications-verbose.tsv', sep='\t')
DOID_NAME = _ind.drop_duplicates('doid_code').set_index('doid_code')['doid_name'].to_dict()
DRUG_NAME = _ind.drop_duplicates('drugbank_id').set_index('drugbank_id')['drugbank_name'].to_dict()

for d in LIPID_DOIDS:
    assert d in doids_in_gold_standard, d
    print(d, '->', DOID_NAME.get(d))

gold standard: (998, 3), 87 DOIDs
DOID:1936 -> atherosclerosis
DOID:3393 -> coronary artery disease


# Load trait → DOID mapping files (same as NB06/NB07)

In [4]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv', sep='\t', index_col='ukb_fullcode')
# Normalize index hyphen->underscore (mirrors NB06/07)
ukb_efo.index = [idx.replace('-', '_', 1) for idx in ukb_efo.index]

efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

# Helper functions (reused from the pipeline)

In [5]:
import sys
sys.path.insert(0, str(here('libs')))
from drug_disease_utils import map_traits_to_doid  # reuse the exact pipeline mapping


def score_matrix(drug_data, disease_data):
    """Pipeline scoring (all-genes/all-LVs, no top-N): score = -1 * drugᵀ · disease.

    drug_data: (k x drugs), disease_data: (k x traits)  ->  (drugs x traits).
    Copied from `predict_dotprod_neg` (sign convention is load-bearing -- do not flip).
    """
    common = drug_data.index.intersection(disease_data.index)
    return -1.0 * drug_data.loc[common].T.dot(disease_data.loc[common])


def build_trait_to_doid(traits, preferred_doids, ukb_efo, efo_xrefs, do_xrefs):
    """Return {trait: DOID} using the SAME algorithm as map_traits_to_doid, so we can
    invert it to learn which UKB trait(s) feed each DOID (needed by the null in NB01).
    Consistency with map_traits_to_doid is asserted below."""
    doid_efo = efo_xrefs[efo_xrefs['target_id_type'] == 'DOID']
    trait_to_doid = {}
    for trait in traits:
        if trait not in ukb_efo.index:
            continue
        rows = ukb_efo.loc[trait]
        if isinstance(rows, pd.Series):
            rows = rows.to_frame().T
        efos = set()
        for tc in rows['term_codes'].dropna():
            for c in str(tc).split(','):
                c = c.strip()
                if c:
                    efos.add(c)
        doids = set()
        for e in efos:
            doids.update(doid_efo[doid_efo['term_id'] == e]['target_id'].values)
            if e.startswith('EFO:'):
                m = (do_xrefs['resource'] == 'EFO') & (do_xrefs['resource_id'] == e[4:])
                doids.update(do_xrefs[m]['doid_code'].values)
        if not doids:
            continue
        pref = sorted(doids & preferred_doids)
        trait_to_doid[trait] = pref[0] if pref else sorted(doids)[0]
    return trait_to_doid

# Load matrices (single-gene gene-level + module-based LV-level)

In [6]:
# Single-gene: gene x drugs (LINCS) and gene x traits (S-PrediXcan Liver)
lincs_gene = pd.read_pickle(LINCS_RAW_FILE)
liver_gene = pd.read_pickle(SPREDIXCAN_RAW_LIVER)
print('LINCS gene-level:', lincs_gene.shape, '| Liver gene-level:', liver_gene.shape)

# Module-based: LV x drugs (LINCS projection) and LV x traits (S-PrediXcan Liver projection)
lincs_proj = pd.read_pickle(LINCS_PROJ_FILE)
liver_proj = pd.read_pickle(SPREDIXCAN_PROJ_LIVER)
print('LINCS projection:', lincs_proj.shape, '| Liver projection:', liver_proj.shape)
assert liver_proj.index.equals(lincs_proj.index)  # both LV-indexed (mirrors NB07)

# Pack drug (L) and disease (D) matrices per method
L = {'gene_based': lincs_gene, 'module_based_archs4': lincs_proj}
D = {'gene_based': liver_gene, 'module_based_archs4': liver_proj}

LINCS gene-level: (7120, 1170) | Liver gene-level: (12025, 4091)
LINCS projection: (1728, 1170) | Liver projection: (1728, 4091)


# Observed scores (drugs × traits) → DOID

In [7]:
# trait-level scores: drugs x traits, per method
scores_trait = {m: score_matrix(L[m], D[m]) for m in METHODS}
for m in METHODS:
    print(m, 'scores (drugs x traits):', scores_trait[m].shape)

# DOID-level scores via the exact pipeline mapping (drugs x DOID; max over traits per DOID)
scores_doid = {
    m: map_traits_to_doid(scores_trait[m], doids_in_gold_standard, ukb_efo, efo_xrefs, do_xrefs)
    for m in METHODS}
for m in METHODS:
    print(m, 'scores (drugs x DOID):', scores_doid[m].shape)

gene_based scores (drugs x traits): (1170, 4091)
module_based_archs4 scores (drugs x traits): (1170, 4091)


gene_based scores (drugs x DOID): (1170, 364)
module_based_archs4 scores (drugs x DOID): (1170, 364)


# Which UKB trait(s) feed each lipid DOID (for the null in NB01)

In [8]:
# Build & verify the trait->DOID dict is consistent with map_traits_to_doid's column set
trait_to_doid = build_trait_to_doid(
    scores_trait['gene_based'].columns, doids_in_gold_standard, ukb_efo, efo_xrefs, do_xrefs)
assert set(trait_to_doid.values()) == set(scores_doid['gene_based'].columns), \
    'build_trait_to_doid diverged from map_traits_to_doid'

contrib_traits = {}
for d in LIPID_DOIDS:
    contrib_traits[d] = sorted(t for t, dd in trait_to_doid.items() if dd == d)
    print(d, DOID_NAME.get(d), '<- traits:', contrib_traits[d])

DOID:1936 atherosclerosis <- traits: ['I70_Diagnoses_main_ICD10_I70_Atherosclerosis']
DOID:3393 coronary artery disease <- traits: ['20002_1075_Noncancer_illness_code_selfreported_heart_attackmyocardial_infarction', 'CARDIoGRAM_C4D_CAD_ADDITIVE', 'I25_Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease']


# Current top drugs + where the gold-standard labels land

In [9]:
def annotate(method, doid, top_n=20):
    s = scores_doid[method][doid].sort_values(ascending=False)
    ranks = s.rank(ascending=False, method='min')
    pct = s.rank(ascending=True, pct=True)  # higher score -> higher percentile
    gs = gold_standard[gold_standard['trait'] == doid].set_index('drug')['true_class']
    out = pd.DataFrame({
        'drug': s.index,
        'drug_name': [DRUG_NAME.get(x, x) for x in s.index],
        'score': s.values,
        'rank': ranks.loc[s.index].values.astype(int),
        'pct': pct.loc[s.index].values,
        'true_class': [gs.get(x, np.nan) for x in s.index],
    })
    return out

for m in METHODS:
    for d in LIPID_DOIDS:
        a = annotate(m, d)
        print(f'\n===== {m} | {d} {DOID_NAME.get(d)} =====')
        print('TOP 15 by raw score:')
        display(a.head(15))
        labeled = a.dropna(subset=['true_class'])
        print(f'gold-standard labeled drugs: {len(labeled)} '
              f'(pos={int((labeled.true_class==1).sum())}, neg={int((labeled.true_class==0).sum())})')
        display(labeled.sort_values('rank')[['drug_name','score','rank','pct','true_class']])


===== gene_based | DOID:1936 atherosclerosis =====
TOP 15 by raw score:


,drug,drug_name,score,rank,pct,true_class
0,DB04297,DB04297,2159.989651,1,1.000000,NaN
1,DB02546,Vorinostat,1815.124127,2,0.999145,NaN
2,DB00390,Digoxin,1648.321005,3,0.998291,NaN
3,DB01092,DB01092,1564.163430,4,0.997436,NaN
4,DB04177,DB04177,1244.713008,5,0.996581,NaN
5,DB01396,DB01396,1130.614312,6,0.995726,NaN
6,DB04865,DB04865,1091.283474,7,0.994872,NaN
7,DB08597,DB08597,821.421808,8,0.994017,NaN
8,DB01735,DB01735,802.184858,9,0.993162,NaN
9,DB07374,DB07374,777.516410,10,0.992308,NaN


gold-standard labeled drugs: 8 (pos=6, neg=2)


,drug_name,score,rank,pct,true_class
64,Clopidogrel,170.892631,65,0.945299,0.0
123,Niacin,112.594340,124,0.894872,1.0
641,Ezetimibe,-5.019970,642,0.452137,1.0
705,Tolazoline,-15.804485,706,0.397436,0.0
956,Pravastatin,-61.705023,957,0.182906,1.0
1140,Lovastatin,-159.390633,1141,0.025641,1.0
1153,Rosuvastatin,-232.453843,1154,0.014530,1.0
1159,Simvastatin,-334.836936,1160,0.009402,1.0



===== gene_based | DOID:3393 coronary artery disease =====
TOP 15 by raw score:


...

,drug_name,score,rank,pct,true_class
93,Pravastatin,0.011952,94,0.920513,1.0
659,Niacin,-0.002504,660,0.436752,1.0
689,Clopidogrel,-0.003031,690,0.411111,0.0
879,Tolazoline,-0.007302,880,0.248718,0.0
1084,Ezetimibe,-0.018018,1085,0.073504,1.0
1118,Rosuvastatin,-0.024331,1119,0.044444,1.0
1146,Lovastatin,-0.042993,1147,0.020513,1.0
1150,Simvastatin,-0.046503,1151,0.017094,1.0



===== module_based_archs4 | DOID:3393 coronary artery disease =====
TOP 15 by raw score:


,drug,drug_name,score,rank,pct,true_class
0,DB00997,Doxorubicin,0.195234,1,0.999573,NaN
1,DB00445,Epirubicin,0.195234,1,0.999573,NaN
2,DB00694,Daunorubicin,0.179604,3,0.998291,NaN
3,DB03496,DB03496,0.177784,4,0.997436,NaN
4,DB08059,DB08059,0.169317,5,0.996581,NaN
5,DB00877,Sirolimus,0.142815,6,0.995726,NaN
6,DB01204,Mitoxantrone,0.133546,7,0.994872,NaN
7,DB02010,DB02010,0.129461,8,0.994017,NaN
8,DB08597,DB08597,0.121296,9,0.993162,NaN
9,DB08142,DB08142,0.115055,10,0.992308,NaN


gold-standard labeled drugs: 28 (pos=22, neg=6)


,drug_name,score,rank,pct,true_class
33,Simvastatin,0.052099,34,0.971795,1.0
61,Lovastatin,0.035348,62,0.947863,1.0
105,Rosuvastatin,0.026751,106,0.910256,1.0
109,Amiodarone,0.025712,110,0.906838,0.0
115,Atorvastatin,0.025193,116,0.901709,1.0
236,Ezetimibe,0.017207,237,0.798291,1.0
280,Niacin,0.015254,281,0.760684,1.0
342,Enalapril,0.013514,343,0.707692,1.0
383,Lisinopril,0.012368,384,0.672650,1.0
399,Valsartan,0.011930,400,0.658974,1.0


# Save outputs for NB01 / NB02

In [10]:
# (a) long observed frame: every drug x method x lipid DOID, with gold-standard label (NaN if none)
rows = []
for m in METHODS:
    for d in LIPID_DOIDS:
        a = annotate(m, d)
        a.insert(0, 'method', m)
        a.insert(1, 'doid', d)
        rows.append(a)
observed = pd.concat(rows, ignore_index=True)
observed.to_pickle(OUTPUT_DIR / 'observed_lipid_scores.pkl')
print('observed_lipid_scores.pkl:', observed.shape)

# (b) full drugs x DOID matrices (needed by the background-trait null in NB01)
pd.to_pickle(scores_doid, OUTPUT_DIR / 'observed_doid_matrices.pkl')

# (c) which traits feed each lipid DOID (small json)
import json
(OUTPUT_DIR / 'contrib_traits.json').write_text(json.dumps(contrib_traits, indent=2))
print('saved observed_doid_matrices.pkl + contrib_traits.json')
display(observed.head())

observed_lipid_scores.pkl: (4680, 8)
saved observed_doid_matrices.pkl + contrib_traits.json


,method,doid,drug,drug_name,score,rank,pct,true_class
0,gene_based,DOID:1936,DB04297,DB04297,2159.989651,1,1.000000,NaN
1,gene_based,DOID:1936,DB02546,Vorinostat,1815.124127,2,0.999145,NaN
2,gene_based,DOID:1936,DB00390,Digoxin,1648.321005,3,0.998291,NaN
3,gene_based,DOID:1936,DB01092,DB01092,1564.163430,4,0.997436,NaN
4,gene_based,DOID:1936,DB04177,DB04177,1244.713008,5,0.996581,NaN
